##  Imports & All Definitions (Everything in One Cell)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import random
import time
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# ── Paths ──────────────────────────────────────────────────
PREPROCESSED  = Path(r"C:\Users\Admin\Desktop\madhura\nnunet_project\preprocessed")
PREP_IMGS_TR  = PREPROCESSED / "imagesTr"
PREP_LBLS_TR  = PREPROCESSED / "labelsTr"
RESULTS_PATH  = Path(r"C:\Users\Admin\Desktop\madhura\nnunet_project\results\05_training")
CHECKPOINT_PATH = Path(r"C:\Users\Admin\Desktop\madhura\nnunet_project\checkpoints")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Hyperparameters ────────────────────────────────────────
PATCH_SIZE   = (128, 128, 128)
BATCH_SIZE   = 2
NUM_EPOCHS   = 100
LR           = 1e-3
PATIENCE     = 20          # early stopping patience

TRAIN_CASES  = ['la_003','la_004','la_005','la_007','la_011','la_014',
                'la_016','la_017','la_018','la_019','la_020','la_021',
                'la_023','la_024','la_026','la_029']
VAL_CASES    = ['la_009','la_010','la_022','la_030']

print("✅ Imports done")
print(f"✅ Device : {device} ({torch.cuda.get_device_name(0)})")
print(f"✅ Paths  : results → {RESULTS_PATH}")
print(f"           checkpoints → {CHECKPOINT_PATH}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\Admin\.conda\envs\nnunetv2env\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Admin\.conda\envs\nnunetv2env\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\Admin\.conda\envs\nnunetv2env\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Admin\.conda\envs\nnunetv2env\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    a

✅ Imports done
✅ Device : cuda (NVIDIA GeForce RTX 4060)
✅ Paths  : results → C:\Users\Admin\Desktop\madhura\nnunet_project\results\05_training
           checkpoints → C:\Users\Admin\Desktop\madhura\nnunet_project\checkpoints


## Dataset, Model & Loss Definitions

In [2]:
# ══════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════
class AtriumDataset(Dataset):
    def __init__(self, case_ids, img_dir, lbl_dir,
                 patch_size=PATCH_SIZE, mode="train"):
        self.case_ids   = case_ids
        self.patch_size = patch_size
        self.mode       = mode
        self.images, self.labels = [], []
        for case_id in case_ids:
            self.images.append(np.load(Path(img_dir) / f"{case_id}.npy"))
            self.labels.append(np.load(Path(lbl_dir) / f"{case_id}.npy"))

    def __len__(self):
        return len(self.case_ids) * (10 if self.mode == "train" else 1)

    def _random_patch(self, img, lbl):
        D, H, W    = img.shape
        pd, ph, pw = self.patch_size
        atrium     = np.argwhere(lbl == 1)
        if len(atrium) > 0 and random.random() < 0.5:
            cd, ch, cw = atrium[random.randint(0, len(atrium)-1)]
        else:
            cd = random.randint(pd//2, D-pd//2)
            ch = random.randint(ph//2, H-ph//2)
            cw = random.randint(pw//2, W-pw//2)
        d0 = max(0, min(cd-pd//2, D-pd))
        h0 = max(0, min(ch-ph//2, H-ph))
        w0 = max(0, min(cw-pw//2, W-pw))
        return img[d0:d0+pd, h0:h0+ph, w0:w0+pw], \
               lbl[d0:d0+pd, h0:h0+ph, w0:w0+pw]

    def _augment(self, img, lbl):
        for axis in range(3):
            if random.random() < 0.5:
                img = np.flip(img, axis=axis).copy()
                lbl = np.flip(lbl, axis=axis).copy()
        img = img * random.uniform(0.9, 1.1)
        return img, lbl

    def _center_patch(self, img, lbl):
        D, H, W    = img.shape
        pd, ph, pw = self.patch_size
        d0 = (D-pd)//2
        h0 = (H-ph)//2
        w0 = (W-pw)//2
        return img[d0:d0+pd, h0:h0+ph, w0:w0+pw], \
               lbl[d0:d0+pd, h0:h0+ph, w0:w0+pw]

    def __getitem__(self, idx):
        vol_idx = idx % len(self.case_ids)
        img = self.images[vol_idx].copy()
        lbl = self.labels[vol_idx].copy()
        if self.mode == "train":
            img, lbl = self._random_patch(img, lbl)
            img, lbl = self._augment(img, lbl)
        else:
            img, lbl = self._center_patch(img, lbl)
        img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
        lbl = torch.tensor(lbl, dtype=torch.long)
        return img, lbl


# ══════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════
class DoubleConv3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNet3D(nn.Module):
    def __init__(self, in_channels=1, num_classes=2,
                 features=[32, 64, 128, 256]):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.encoders.append(DoubleConv3D(ch, f))
            self.pools.append(nn.MaxPool3d(2, 2))
            ch = f
        self.bottleneck = DoubleConv3D(features[-1], features[-1]*2)
        self.upconvs    = nn.ModuleList()
        self.decoders   = nn.ModuleList()
        ch = features[-1] * 2
        for f in reversed(features):
            self.upconvs.append(nn.ConvTranspose3d(ch, f, 2, stride=2))
            self.decoders.append(DoubleConv3D(f*2, f))
            ch = f
        self.output_conv = nn.Conv3d(features[0], num_classes, 1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.upconvs, self.decoders, skips[::-1]):
            x = up(x)
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])
            x = dec(torch.cat([skip, x], dim=1))
        return self.output_conv(x)


# ══════════════════════════════════════════════════════════
# LOSS
# ══════════════════════════════════════════════════════════
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
    def forward(self, preds, targets):
        probs    = F.softmax(preds, dim=1)[:, 1]
        targets_f = targets.float()
        intersection = (probs * targets_f).sum()
        return 1.0 - (2.0*intersection + self.smooth) / \
               (probs.sum() + targets_f.sum() + self.smooth)

class DiceCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice = DiceLoss()
        self.ce   = nn.CrossEntropyLoss()
    def forward(self, preds, targets):
        d = self.dice(preds, targets)
        c = self.ce(preds, targets)
        return d + c, d, c


print("✅ AtriumDataset defined")
print("✅ UNet3D defined")
print("✅ DiceCELoss defined")

✅ AtriumDataset defined
✅ UNet3D defined
✅ DiceCELoss defined


## Instantiate Model, Optimizer & Scheduler

In [3]:
# ── Datasets & Loaders ─────────────────────────────────────
print("Loading datasets...")
train_dataset = AtriumDataset(TRAIN_CASES, PREP_IMGS_TR, PREP_LBLS_TR, mode="train")
val_dataset   = AtriumDataset(VAL_CASES,   PREP_IMGS_TR, PREP_LBLS_TR, mode="val")

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=0, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=1,
                           shuffle=False, num_workers=0, pin_memory=True)

# ── Model ──────────────────────────────────────────────────
model     = UNet3D(in_channels=1, num_classes=2).to(device)
criterion = DiceCELoss()

# ── Optimizer (Adam — nnU-Net style) ───────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=LR,
                             weight_decay=1e-5)

# ── Polynomial LR Scheduler (nnU-Net style) ────────────────
# LR decays as (1 - epoch/max_epochs)^0.9
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda epoch: (1 - epoch / NUM_EPOCHS) ** 0.9
)

print("\n✅ Everything instantiated")
print(f"   Model params   : {sum(p.numel() for p in model.parameters()):,}")
print(f"   Optimizer      : Adam (lr={LR}, wd=1e-5)")
print(f"   Scheduler      : Polynomial decay (power=0.9)")
print(f"   Train batches  : {len(train_loader)}/epoch")
print(f"   Val batches    : {len(val_loader)}/epoch")
print(f"   Epochs         : {NUM_EPOCHS}")
print(f"   Early stopping : patience={PATIENCE}")

Loading datasets...

✅ Everything instantiated
   Model params   : 22,578,306
   Optimizer      : Adam (lr=0.001, wd=1e-5)
   Scheduler      : Polynomial decay (power=0.9)
   Train batches  : 80/epoch
   Val batches    : 4/epoch
   Epochs         : 100
   Early stopping : patience=20


## Training & Validation Functions

In [4]:
def dice_score(preds, targets, smooth=1e-5):
    """Compute Dice score for evaluation."""
    probs    = F.softmax(preds, dim=1)[:, 1]
    pred_bin = (probs > 0.5).float()
    targets_f = targets.float()
    intersection = (pred_bin * targets_f).sum()
    return ((2.0 * intersection + smooth) /
            (pred_bin.sum() + targets_f.sum() + smooth)).item()


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    epoch_loss = 0.0
    epoch_dice = 0.0
    epoch_ce   = 0.0

    for img, lbl in loader:
        img = img.to(device, non_blocking=True)
        lbl = lbl.to(device, non_blocking=True)

        optimizer.zero_grad()
        preds = model(img)
        loss, dice_l, ce_l = criterion(preds, lbl)
        loss.backward()
        # Gradient clipping (nnU-Net style)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=12)
        optimizer.step()

        epoch_loss += loss.item()
        epoch_dice += dice_l.item()
        epoch_ce   += ce_l.item()

    n = len(loader)
    return epoch_loss/n, epoch_dice/n, epoch_ce/n


def validate(model, loader, criterion, device):
    model.eval()
    val_loss  = 0.0
    val_dice  = 0.0

    with torch.no_grad():
        for img, lbl in loader:
            img = img.to(device, non_blocking=True)
            lbl = lbl.to(device, non_blocking=True)
            preds = model(img)
            loss, _, _ = criterion(preds, lbl)
            val_loss  += loss.item()
            val_dice  += dice_score(preds, lbl)

    n = len(loader)
    return val_loss/n, val_dice/n


print("✅ train_one_epoch() defined")
print("✅ validate() defined")
print("✅ dice_score() defined")
print("\nReady to start training!")
print(f"Expected time per epoch: ~2-4 min on RTX 4060")
print(f"Total estimated time   : ~{100*3//60}–{100*4//60} hrs for 100 epochs")

✅ train_one_epoch() defined
✅ validate() defined
✅ dice_score() defined

Ready to start training!
Expected time per epoch: ~2-4 min on RTX 4060
Total estimated time   : ~5–6 hrs for 100 epochs


## Training Loop

In [5]:
# ── Training history ───────────────────────────────────────
history = {
    "train_loss": [], "train_dice": [], "train_ce": [],
    "val_loss"  : [], "val_dice"  : [], "lr"       : []
}

best_val_dice    = 0.0
patience_counter = 0
best_epoch       = 0

print("=" * 65)
print("  STARTING TRAINING")
print("=" * 65)
print(f"  Epochs     : {NUM_EPOCHS}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Device     : {device}")
print(f"  Checkpoint : {CHECKPOINT_PATH / 'best_model.pth'}")
print("=" * 65)

total_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()
    current_lr  = optimizer.param_groups[0]["lr"]

    # ── Train ─────────────────────────────────────────────
    train_loss, train_dice, train_ce = train_one_epoch(
        model, train_loader, optimizer, criterion, device)

    # ── Validate ──────────────────────────────────────────
    val_loss, val_dice = validate(model, val_loader, criterion, device)

    # ── Scheduler step ────────────────────────────────────
    scheduler.step()

    # ── Save history ──────────────────────────────────────
    history["train_loss"].append(train_loss)
    history["train_dice"].append(train_dice)
    history["train_ce"].append(train_ce)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)
    history["lr"].append(current_lr)

    epoch_time = time.time() - epoch_start

    # ── Checkpoint best model ─────────────────────────────
    if val_dice > best_val_dice:
        best_val_dice    = val_dice
        best_epoch       = epoch
        patience_counter = 0
        torch.save({
            "epoch"          : epoch,
            "model_state"    : model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_dice"  : best_val_dice,
            "history"        : history,
        }, CHECKPOINT_PATH / "best_model.pth")
        ckpt_tag = " ← ✅ BEST"
    else:
        patience_counter += 1
        ckpt_tag = f" (patience {patience_counter}/{PATIENCE})"

    # ── Print progress ────────────────────────────────────
    print(f"Epoch [{epoch:03d}/{NUM_EPOCHS}] "
          f"| LR={current_lr:.6f} "
          f"| Train Loss={train_loss:.4f} "
          f"| Train Dice={1-train_dice:.4f} "
          f"| Val Loss={val_loss:.4f} "
          f"| Val Dice={val_dice:.4f} "
          f"| {epoch_time:.0f}s"
          f"{ckpt_tag}")

    # ── Early stopping ────────────────────────────────────
    if patience_counter >= PATIENCE:
        print(f"\n⚠️  Early stopping at epoch {epoch}")
        print(f"   Best val Dice: {best_val_dice:.4f} at epoch {best_epoch}")
        break

total_time = time.time() - total_start
print(f"\n{'='*65}")
print(f"  TRAINING COMPLETE")
print(f"  Total time     : {total_time/3600:.2f} hrs")
print(f"  Best Val Dice  : {best_val_dice:.4f} at epoch {best_epoch}")
print(f"  Checkpoint     : {CHECKPOINT_PATH / 'best_model.pth'}")
print(f"{'='*65}")

  STARTING TRAINING
  Epochs     : 100
  Batch size : 2
  Device     : cuda
  Checkpoint : C:\Users\Admin\Desktop\madhura\nnunet_project\checkpoints\best_model.pth
Epoch [001/100] | LR=0.001000 | Train Loss=1.2280 | Train Dice=0.0984 | Val Loss=0.9526 | Val Dice=0.6322 | 1808s ← ✅ BEST
Epoch [002/100] | LR=0.000991 | Train Loss=0.8979 | Train Dice=0.2090 | Val Loss=0.6163 | Val Dice=0.6679 | 1804s ← ✅ BEST
Epoch [003/100] | LR=0.000982 | Train Loss=0.7454 | Train Dice=0.3271 | Val Loss=0.4746 | Val Dice=0.6661 | 1804s (patience 1/20)
Epoch [004/100] | LR=0.000973 | Train Loss=0.5645 | Train Dice=0.4913 | Val Loss=0.4894 | Val Dice=0.6080 | 1803s (patience 2/20)
Epoch [005/100] | LR=0.000964 | Train Loss=0.5131 | Train Dice=0.5386 | Val Loss=0.2689 | Val Dice=0.7953 | 1803s ← ✅ BEST
Epoch [006/100] | LR=0.000955 | Train Loss=0.3590 | Train Dice=0.6810 | Val Loss=0.2731 | Val Dice=0.7809 | 1803s (patience 1/20)
Epoch [007/100] | LR=0.000946 | Train Loss=0.4355 | Train Dice=0.6119 | Val L